# auth

> Log in to Hugging Face, Kaggle, Colab and AWS from a notebook or a terminal: `login("hf")`, `login("kaggle")`, `login("colab")`, `login("aws", local=True)`, and `whoami()` to see which accounts are in use.

In [ ]:
#| default_exp auth

In [ ]:
#| export
import base64, datetime, hashlib, json, os, re, secrets, shutil, subprocess, sys, urllib.parse, urllib.request
from pathlib import Path

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import tempfile

Each service keeps its own login, and each has a sign-in that works without typing a password into a notebook:

| Service | Sign-in | Where the login is kept |
|---|---|---|
| Hugging Face | a browser page with a short code (`huggingface_hub.login`) | `~/.cache/huggingface/token` |
| Kaggle | a browser page, then a code to paste (`kaggle auth login`) | `~/.kaggle/credentials.json` |
| Colab | a Google consent page, then a code to paste (the `colab` CLI) | `~/.config/colab-cli/token.json` |
| AWS | a profile (`aws configure`, or `aws sso login`), or dummy keys for a local emulator | `~/.aws` |

Inside Colab or Kaggle, a login can also come from the notebook's secrets (`HF_TOKEN`, `KAGGLE_API_TOKEN`): it is then kept in this process's environment only, never written to disk. Tokens are never printed; `whoami` shows account names.

In [ ]:
#| export
class AuthError(RuntimeError): "A login failed or a sign-in tool is missing"

def in_colab() -> bool:
    "Whether this runs in a Colab notebook"
    return "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

def in_kaggle() -> bool:
    "Whether this runs in a Kaggle notebook"
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def notebook_secret(name:str) -> str|None:
    "A secret of the notebook service this runs in (Colab's secrets with notebook access on, or Kaggle's), else None"
    try:
        if in_colab():
            from google.colab import userdata
            return userdata.get(name)
        if in_kaggle():
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception: return None
    return None

In [ ]:
test_eq((in_colab(), in_kaggle(), notebook_secret("HF_TOKEN")), (False, False, None))

## Copy-paste sign-ins

Kaggle's and Colab's command-line tools sign in the same way when no browser can call them back: they print a URL, the user signs in there, and the page shows a code to paste back. `paste_code_login` runs such a tool, shows the URL, asks for the code (with `input`, which works in Jupyter as in a terminal), and passes it on. Here it drives a stand-in that prints a URL to stderr, as Colab's tool does:

In [ ]:
#| export
def paste_code_login(cmd:list, code=input, timeout:float=600, hint:str="") -> str:
    "Run a sign-in that prints a URL and waits for a code: show the URL (and `hint`), ask for the code, pass it on, and return the tool's output"
    if shutil.which(cmd[0]) is None: raise AuthError(f"`{cmd[0]}` is not installed (uv tool install {'google-colab-cli' if cmd[0] == 'colab' else cmd[0]})")
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in p.stdout:
        m = re.search(r"https://\S+", line)
        if m: url = m.group(0); break
    if url is None:
        p.wait(); raise AuthError(f"{cmd[0]} printed no sign-in URL")
    print(f"Open this link, sign in, and paste the code the page shows:\n\n  {url}\n" + (f"\n{hint}\n" if hint else ""), flush=True)
    out, _ = p.communicate(code("Code: ").strip() + "\n", timeout=timeout)
    if p.returncode != 0: raise AuthError(f"{' '.join(cmd)} failed: {out.strip()[-500:]}")
    return out

In [ ]:
fake = [sys.executable, "-c", "import sys; print('visit', file=sys.stderr); print('  https://example.com/auth?x=1', file=sys.stderr); "
                              "c = input('Enter the authorization code: '); print('signed in with', c)"]
test_eq(paste_code_login(fake, code=lambda prompt: " abc123 ").strip().endswith("signed in with abc123"), True)
test_fail(lambda: paste_code_login(["no-such-tool-xyz"]), contains="not installed")
narrow = [sys.executable, "-c", "import sys; print('  https://example.com/auth', file=sys.stderr); input('Code: '); "
                               "sys.exit('Warning: Scope has changed from \"a b c\" to \"a\".')"]
test_fail(lambda: paste_code_login(narrow, code=lambda p: "x"), contains="Scope has changed")

## Hugging Face

`hf_login` uses, in order, a token passed in, `HF_TOKEN` (from the environment or a notebook secret), or the login already saved; with none of them it runs the Hub's sign-in, which saves a token for the machine.

In [ ]:
#| export
def hf_user(token:str|None=None) -> str|None:
    "The Hub account in use (from `token`, `HF_TOKEN` or the saved login), or None when there is none or the Hub is out of reach"
    from huggingface_hub import get_token, whoami
    if token is None and get_token() is None: return None
    try: return whoami(token=token)["name"]
    except Exception: return None

def hf_login(token:str|None=None) -> str|None:
    "Log in to the Hub: a token given or found in a notebook secret (kept for this process), the saved login, or the Hub's browser sign-in"
    from huggingface_hub import get_token, login
    token = token or notebook_secret("HF_TOKEN")
    if token: os.environ["HF_TOKEN"] = token
    elif get_token() is None: login()
    return hf_user()

## Kaggle

The Kaggle CLI reads, in order, `KAGGLE_API_TOKEN`, `KAGGLE_USERNAME` with `KAGGLE_KEY`, and the files in `~/.kaggle` (or `KAGGLE_CONFIG_DIR`): `credentials.json` from `kaggle auth login`, `access_token`, or the older `kaggle.json`. `kaggle_credentials` says which one it will use. Inside a Kaggle notebook the API is already authorised.

In [ ]:
#| export
KAGGLE_FILES = ("credentials.json", "access_token", "kaggle.json")

def kaggle_dir() -> Path: return Path(os.environ.get("KAGGLE_CONFIG_DIR") or Path.home() / ".kaggle")

def kaggle_credentials() -> str|None:
    "Where the Kaggle CLI will find credentials, or None if nowhere"
    if os.environ.get("KAGGLE_API_TOKEN"): return "KAGGLE_API_TOKEN"
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"): return "KAGGLE_USERNAME and KAGGLE_KEY"
    for f in KAGGLE_FILES:
        p = kaggle_dir() / f
        if p.exists() and p.stat().st_size > 0: return f"~/.kaggle/{f}" if "KAGGLE_CONFIG_DIR" not in os.environ else f
    return None

def kaggle_user() -> str|None:
    "The Kaggle account the CLI uses, as `kaggle config view` reports it, or None"
    if kaggle_credentials() is None or shutil.which("kaggle") is None: return None
    try: r = subprocess.run(["kaggle", "config", "view"], capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.SubprocessError): return None
    m = re.search(r"username:\s*(\S+)", r.stdout)
    return m.group(1) if m else None

def kaggle_login(token:str|None=None, code=input) -> str|None:
    "Log in to Kaggle: a token given or found in a notebook secret (kept for this process), saved credentials, or the CLI's sign-in"
    token = token or notebook_secret("KAGGLE_API_TOKEN")
    if token: os.environ["KAGGLE_API_TOKEN"] = token
    elif in_kaggle(): return "kaggle notebook"
    elif kaggle_credentials() is None: paste_code_login(["kaggle", "auth", "login", "--no-launch-browser"], code)
    return kaggle_user() or kaggle_credentials()

In [ ]:
kd = Path(tempfile.mkdtemp())
saved = {k: os.environ.pop(k, None) for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_CONFIG_DIR")}
os.environ["KAGGLE_CONFIG_DIR"] = str(kd)
test_eq(kaggle_credentials(), None)
(kd / "kaggle.json").write_text('{"username": "me", "key": "k"}')
test_eq(kaggle_credentials(), "kaggle.json")
os.environ["KAGGLE_API_TOKEN"] = "KGAT_x"
test_eq(kaggle_credentials(), "KAGGLE_API_TOKEN")
for k, v in saved.items():
    os.environ.pop(k, None)
    if v is not None: os.environ[k] = v

## Colab

The Colab CLI signs in on first use with `--auth=oauth2` (Google's consent page shows a code to paste back) and keeps the token in `~/.config/colab-cli/token.json`. It asks for six permissions: your identity, Colab, Google Cloud (for `colab auth`, which gives a VM access to GCS or BigQuery) and the Drive files it opens (for `colab drivemount`). The page lets you untick any of them, but the CLI then refuses the sign-in ("Scope has changed"), and it renews its token asking for all six again.

Running jobs needs only two of them: your email, which Colab's session service checks, and Colab itself. So `colab_login` signs in with those alone. It runs the same copy-paste flow as the CLI, with the same public client the CLI ships, and writes the token where the CLI reads it. The CLI uses any token that has not expired, whatever its permissions, but renewing one would ask Google for all six, which it was not granted. So sysone renews the token itself, with the permissions it has, before every Colab command it runs (`refresh_colab_token`, called by [colab](45_colab.ipynb)); running `sysone login colab` again does the same, before using the `colab` CLI by hand. For the same reason the CLI's hidden `colab whoami`, which always renews, fails on such a token; `colab_user` asks Google whose token it is. `colab_login(full=True)` signs in through the CLI instead, with all six permissions, for `colab auth` and `colab drivemount`.

In [ ]:
#| export
def colab_token() -> Path: return Path(os.environ.get("SYSONE_COLAB_TOKEN") or Path.home() / ".config" / "colab-cli" / "token.json")

COLAB_SCOPES = ("openid", "https://www.googleapis.com/auth/userinfo.email", "https://www.googleapis.com/auth/colaboratory")
COLAB_REDIRECT = "https://sdk.cloud.google.com/applicationdefaultauthcode.html"    # the page that shows the code, as for the CLI
GOOGLE_AUTH_URI, GOOGLE_TOKEN_URI = "https://accounts.google.com/o/oauth2/auth", "https://oauth2.googleapis.com/token"
GOOGLE_USERINFO_URI = "https://openidconnect.googleapis.com/v1/userinfo"

def colab_client() -> dict:
    "The OAuth client the Colab CLI signs in with: ~/.colab-cli-oauth-config.json if you made one, else the one the CLI ships"
    own = Path.home() / ".colab-cli-oauth-config.json"
    if own.exists(): cfg = json.loads(own.read_text())
    else:
        exe = shutil.which("colab")
        if exe is None: raise AuthError("the Colab CLI is not installed: uv tool install google-colab-cli")
        py = open(exe).readline()[2:].strip()           # the CLI's own interpreter, from its script's first line
        r = subprocess.run([py, "-c", "import colab_cli, os; print(os.path.join(list(colab_cli.__path__)[0], 'oauth_config.json'))"],
                           capture_output=True, text=True, timeout=60)     # __path__: colab_cli is a namespace package (no __file__)
        if r.returncode != 0: raise AuthError(f"could not find the Colab CLI's OAuth client: {r.stderr[-300:]}")
        cfg = json.loads(Path(r.stdout.strip()).read_text())
    return cfg.get("installed") or cfg.get("web") or cfg

def _token_request(data:dict) -> dict:
    "POST to Google's token endpoint; the JSON answer"
    req = urllib.request.Request(GOOGLE_TOKEN_URI, data=urllib.parse.urlencode(data).encode(),
                                 headers={"Content-Type": "application/x-www-form-urlencoded"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r: return json.loads(r.read())
    except urllib.error.HTTPError as e:
        raise AuthError(f"Google's token endpoint refused: {e.read().decode(errors='replace')[:300]}") from None

def _expiry(seconds) -> str:
    t = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None) + datetime.timedelta(seconds=int(seconds))
    return t.strftime("%Y-%m-%dT%H:%M:%SZ")

def _write_token(info:dict):
    "Write the token where the CLI reads it, readable by you alone"
    p = colab_token(); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(".tmp")
    tmp.write_text(json.dumps(info)); tmp.chmod(0o600); tmp.replace(p)

def colab_scoped_login(code=input, scopes:tuple=COLAB_SCOPES) -> list:
    "Sign in with only `scopes` (by default your email and Colab), through the CLI's copy-paste page; returns the granted scopes"
    c = colab_client()
    verifier = secrets.token_urlsafe(64)
    challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b"=").decode()
    q = {"response_type": "code", "client_id": c["client_id"], "redirect_uri": COLAB_REDIRECT, "scope": " ".join(scopes),
         "access_type": "offline", "prompt": "consent", "token_usage": "remote", "code_challenge": challenge, "code_challenge_method": "S256"}
    print(f"Open this link, sign in, and paste the code the page shows:\n\n  {GOOGLE_AUTH_URI}?{urllib.parse.urlencode(q)}\n\n"
          "It asks for your email and for Colab only.\n", flush=True)
    r = _token_request({"grant_type": "authorization_code", "code": code("Code: ").strip(), "client_id": c["client_id"],
                        "client_secret": c["client_secret"], "redirect_uri": COLAB_REDIRECT, "code_verifier": verifier})
    granted = r.get("scope", " ".join(scopes)).split()
    missing = [s for s in scopes if s.startswith("https://") and s not in granted]
    if missing: raise AuthError(f"Google did not grant {', '.join(missing)}: sign in again and allow it")
    if not r.get("refresh_token"): raise AuthError("Google sent no refresh token; sign in again")
    _write_token({"token": r["access_token"], "refresh_token": r["refresh_token"], "token_uri": GOOGLE_TOKEN_URI, "client_id": c["client_id"],
                  "client_secret": c["client_secret"], "scopes": granted, "universe_domain": "googleapis.com", "account": "",
                  "expiry": _expiry(r.get("expires_in", 3600))})
    return granted

def refresh_colab_token(margin:float=1200.0) -> bool:
    "Renew the Colab CLI's token, with the permissions it was granted, when it expires within `margin` seconds; True if renewed"
    p = colab_token()
    if not p.exists(): return False
    t = json.loads(p.read_text())
    if t.get("expiry"):
        exp = datetime.datetime.strptime(t["expiry"].rstrip("Z").split(".")[0], "%Y-%m-%dT%H:%M:%S")
        if (exp - datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)).total_seconds() > margin: return False
    r = _token_request({"grant_type": "refresh_token", "refresh_token": t["refresh_token"], "client_id": t["client_id"],
                        "client_secret": t["client_secret"]})     # no scope asked for: the renewal keeps what was granted
    t |= {"token": r["access_token"], "expiry": _expiry(r.get("expires_in", 3600))}
    _write_token(t)
    return True

def _userinfo(token:str) -> dict:
    "Google's OpenID profile for an access token, sent in a header (never in the URL)"
    req = urllib.request.Request(GOOGLE_USERINFO_URI, headers={"Authorization": f"Bearer {token}"})
    with urllib.request.urlopen(req, timeout=30) as r: return json.loads(r.read())

def colab_user() -> str|None:
    "The Google account of the Colab CLI's token (renewed first when it is about to expire), or None when it is not signed in"
    if not colab_token().exists(): return None
    try:
        refresh_colab_token()
        return _userinfo(json.loads(colab_token().read_text())["token"]).get("email")
    except Exception: return None

COLAB_HINT = ("Google asks which permissions to allow: allow all of them (tick every box, or 'Select all'). The Colab CLI asks for "
              "Colab, your email and profile, Google Cloud (for `colab auth`) and the Drive files it opens (for `colab drivemount`), "
              "and refuses a sign-in that grants fewer.")

def colab_login(code=input, full:bool=False) -> str|None:
    "Sign the Colab CLI in: your email and Colab only, or (`full=True`) through the CLI with all its permissions; renews an existing token"
    tok = colab_token()
    if tok.exists():
        scopes = json.loads(tok.read_text()).get("scopes") or []
        if full and "https://www.googleapis.com/auth/cloud-platform" not in scopes: tok.unlink()   # a Colab-only token: sign in again, with all
        else:
            try:
                refresh_colab_token()
                return colab_user()
            except AuthError: tok.unlink()                  # revoked, or lapsed for good: sign in again
    if full:
        try: paste_code_login(["colab", "--auth=oauth2", "sessions"], code, hint=COLAB_HINT)
        except AuthError as e:
            if "Scope has changed" not in str(e): raise
            raise AuthError("Google granted fewer permissions than the Colab CLI asks for, so it refused the sign-in. "
                            "Sign in again and, on Google's consent page, allow every permission ('Select all').") from None
    else: colab_scoped_login(code)
    return colab_user()

In [ ]:
if shutil.which("colab"):         # with the CLI installed: its own client is found (its fields only, nothing printed)
    test_eq({"client_id", "client_secret"} <= set(colab_client()), True)

In [ ]:
os.environ["SYSONE_COLAB_TOKEN"] = str(Path(tempfile.mkdtemp()) / "token.json")
calls, saved = [], (colab_client, _token_request, _userinfo)
colab_client = lambda: {"client_id": "cid.apps.googleusercontent.com", "client_secret": "csecret"}       # a stand-in client
_token_request = lambda d: calls.append(d) or {"access_token": f"at{len(calls)}", "expires_in": 3599, "refresh_token": "rt",
                                                "scope": " ".join(COLAB_SCOPES)}
granted = colab_scoped_login(code=lambda prompt: " 4/code ")
test_eq((calls[0]["grant_type"], calls[0]["code"], calls[0]["redirect_uri"]), ("authorization_code", "4/code", COLAB_REDIRECT))
tok = json.loads(colab_token().read_text())
test_eq((tok["token"], tok["refresh_token"], tok["scopes"], oct(colab_token().stat().st_mode & 0o777)), ("at1", "rt", list(COLAB_SCOPES), "0o600"))
test_eq(refresh_colab_token(), False)                      # an hour left: nothing to do
tok["expiry"] = _expiry(300); colab_token().write_text(json.dumps(tok))
test_eq(refresh_colab_token(), True)                       # five minutes left: renewed, with no scope asked for
test_eq((calls[-1]["grant_type"], "scope" in calls[-1], json.loads(colab_token().read_text())["token"]), ("refresh_token", False, "at2"))
_userinfo = lambda t: {"email": "me@example.com"} if t == "at2" else {}
test_eq(colab_user(), "me@example.com")                    # whose token: asked of Google, with the renewed token
_token_request = lambda d: {"access_token": "x", "refresh_token": "rt", "scope": "openid https://www.googleapis.com/auth/userinfo.email"}
colab_token().unlink()
test_fail(lambda: colab_scoped_login(code=lambda p: "c"), contains="did not grant https://www.googleapis.com/auth/colaboratory")
test_eq(colab_user(), None)                                # no token, no account
del os.environ["SYSONE_COLAB_TOKEN"]; colab_client, _token_request, _userinfo = saved

## AWS

AWS logins are profiles: `aws configure` for keys, `aws sso login --profile NAME` for single sign-on, then `aws_login(profile="NAME")`. For a local emulator such as [floci](https://floci.io), `aws_login(local=True)` points every AWS client of this process at it: `AWS_ENDPOINT_URL`, a region, and dummy keys, which the emulator accepts.

In [ ]:
#| export
FLOCI = "http://localhost:4566"

def aws_local(endpoint:str=FLOCI, region:str="us-east-1") -> dict:
    "Point this process's AWS clients (boto3, the SageMaker SDK) at a local emulator, with dummy keys"
    env = {"AWS_ENDPOINT_URL": endpoint, "AWS_ACCESS_KEY_ID": "test", "AWS_SECRET_ACCESS_KEY": "test",
           "AWS_DEFAULT_REGION": region, "AWS_REGION": region}
    os.environ.update(env)
    return env

def aws_user() -> str|None:
    "The ARN of the AWS identity in use (needs boto3), or None"
    try:
        import boto3
        return boto3.client("sts").get_caller_identity()["Arn"]
    except Exception: return None

def aws_login(profile:str|None=None, local:bool=False, endpoint:str=FLOCI, region:str|None=None) -> str|None:
    "Use AWS: a local emulator (`local=True`), or a profile (SSO profiles need `aws sso login --profile NAME` first)"
    if local: aws_local(endpoint, region or "us-east-1")
    else:
        if profile: os.environ["AWS_PROFILE"] = profile
        if region: os.environ["AWS_DEFAULT_REGION"] = region
    return aws_user()

In [ ]:
saved = {k: os.environ.get(k) for k in ("AWS_ENDPOINT_URL", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_DEFAULT_REGION", "AWS_REGION")}
test_eq(aws_local()["AWS_ENDPOINT_URL"], "http://localhost:4566")
test_eq(os.environ["AWS_ACCESS_KEY_ID"], "test")
for k, v in saved.items():
    if v is None: os.environ.pop(k, None)
    else: os.environ[k] = v

## login and whoami

`login(service)` is the one call to remember; `whoami()` shows each service's account (or `None`).

In [ ]:
#| export
LOGINS = {"hf": hf_login, "huggingface": hf_login, "kaggle": kaggle_login, "colab": colab_login, "aws": aws_login}

def login(service:str="hf", **kw):
    "Log in to `service`: hf, kaggle, colab or aws (see each one's login function for its options)"
    if service.lower() not in LOGINS: raise ValueError(f"unknown service {service!r}; use one of {sorted(set(LOGINS) - {'huggingface'})}")
    return LOGINS[service.lower()](**kw)

def whoami() -> dict:
    "The account each service uses here, or None where there is no login"
    return {"hf": hf_user(), "kaggle": kaggle_user() or kaggle_credentials(), "colab": colab_user(), "aws": aws_user()}

In [ ]:
test_fail(lambda: login("dropbox"), contains="unknown service")

With logins in place (these cells are flagged, so the tests skip them):

In [ ]:
#| hub
test_ne(hf_user(), None)

In [ ]:
#| kaggle
test_ne(kaggle_login(), None)

In [ ]:
#| colab
test_ne(colab_user(), None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()